In [1]:
import sys
import os

# Define paths dynamically
base_dir_str: str = os.path.abspath("")  
data_rel_path_str: str = os.path.join("..", "data") 
data_abs_path_str: str = os.path.normpath(os.path.join(base_dir_str, data_rel_path_str))

# Add the absolute path to sys.path to ensure module imports work correctly
if data_abs_path_str not in sys.path:
    sys.path.append(data_abs_path_str)

# Import your custom dataset
try:
    import dataset
    dataset_imported_bol: bool = True
except ImportError as e:
    print(f"Warning: Could not import dataset module from {data_abs_path_str}. Error: {e}")
    dataset_imported_bol: bool = False



In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from typing import List

def get_semantic_chunks_list(
    input_str: str,
    embedding_model_str: str = "sentence-transformers/all-MiniLM-L6-v2"
) -> List[str]:
    """
    Splits a raw string into chunks based on semantic similarity using
    percentile-based breakpoint detection.

    Args:
        input_str (str): The raw text data to be processed.
        embedding_model_str (str): The HuggingFace model used to calculate sentence meanings.

    Returns:
        List[str]: A list of semantically cohesive text chunks.
    """
    # 1. Initialize the embedding model used to "read" the sentences
    embeddings_obj = HuggingFaceEmbeddings(model_name=embedding_model_str)

    # 2. Initialize the Semantic Chunker
    # breakpoint_threshold_type can be "percentile", "standard_deviation", or "interquartile"
    semantic_splitter_obj = SemanticChunker(
        embeddings_obj,
        breakpoint_threshold_type="percentile" #0.95
    )

    # 3. Create the documents (LangChain returns a list of Document objects)
    docs_list = semantic_splitter_obj.create_documents([input_str])

    # 4. Extract the string content
    chunks_list = [doc.page_content for doc in docs_list]
    return chunks_list

# Sample data with distinct topics
raw_text_str = dataset.test

processed_chunks_list = get_semantic_chunks_list(raw_text_str)

print(f"Total semantic chunks created: {len(processed_chunks_list)}")
for i_int, text_str in enumerate(processed_chunks_list):
    print(f"\n--- Chunk {i_int + 1} ---\n{text_str.strip()}")

C:\Users\sjoshi06\AppData\Local\Temp\ipykernel_18360\2018486746.py:21: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_obj = HuggingFaceEmbeddings(model_name=embedding_model_str)
c:\SJ\SD_tasks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 503.67it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
-----

Total semantic chunks created: 8

--- Chunk 1 ---
Abstract
Background
Cancers arise from multiple acquired mutations, which presumably occur over many years. Early stages in cancer development might be present years before cancers become clinically apparent. Methods
We analyzed data from whole-exome sequencing of DNA in peripheral-blood cells from 12,380 persons, unselected for cancer or hematologic phenotypes. We identified somatic mutations on the basis of unusual allelic fractions. We used data from Swedish national patient registers to follow health outcomes for 2 to 7 years after DNA sampling. Results
Clonal hematopoiesis with somatic mutations was observed in 10% of persons older than 65 years of age but in only 1% of those younger than 50 years of age. Detectable clonal expansions most frequently involved somatic mutations in three genes (DNMT3A, ASXL1, and TET2) that have previously been implicated in hematologic cancers. Clonal hematopoiesis was a strong risk factor for subseq

In [ ]:
from langchain_community.vectorstores import Chroma

def initialize_semantic_vector_db(chunks_list: List[str]) -> Chroma:
    """
    Stores semantic chunks into a Chroma vector database.

    Args:
        chunks_list (List[str]): List of semantically split text strings.

    Returns:
        Chroma: The initialized vector store.
    """
    embeddings_obj = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") #debug

    vector_db_obj = Chroma.from_texts(
        texts=chunks_list,
        embedding=embeddings_obj,
        collection_name="semantic_best_way_collection"
    )

    return vector_db_obj

def query_semantic_pipeline(query_str: str, vector_db: Chroma, k_int: int = 1) -> List[str]:
    """
    Retrieves the most semantically relevant chunk for a user query.

    Args:
        query_str (str): User's question.
        vector_db (Chroma): The vector database.
        k_int (int): Number of chunks to retrieve.

    Returns:
        List[str]: The top matching text segments.
    """
    results_list = vector_db.similarity_search(query_str, k=k_int)

    #dont return dircet list
    return [doc.page_content for doc in results_list]

In [5]:
# Execution
semantic_db_obj = initialize_semantic_vector_db(processed_chunks_list)
user_query_str = "What is clonal hematopoiesis and how common is it in elderly individuals?"
retrieved_context_list = query_semantic_pipeline(user_query_str, semantic_db_obj)

print(f"Search Result:\n{retrieved_context_list[0]}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 522.12it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Search Result:
Because the biopsy specimen showed all these events at high allelic fractions, we concluded that these mutations arose in a series of subclones (Figure 4C). The patient died 2 months after diagnosis. Discussion
We observed clonal hematopoiesis with somatic mutations in 10% of the elderly study participants, with increasing frequency with age (Figure 2D). Most such clonal expansions appeared to involve driver genes and mutations that are also driver mutations in hematologic cancers (Figure 2A and 2B). We found the presence of such clones to be a risk factor for subsequent hematologic cancers (hazard ratio, 12.9; 95% CI, 5.8 to 28.7) (Figure 3A and 3B) and death (hazard ratio, 1.4; 95% CI, 1.0 to 1.8) (Figure 3D and 3E). Most cases of clonal hematopoiesis appeared to involve mutations in a specific subset of the genes recognized as drivers of blood cancers,22 such as DNMT3A, ASXL1, and TET2 (Figure 2A). Other common mutational drivers of such cancers — for example, activat